In [21]:
# --- Imports ---
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import os
from pathlib import Path

In [22]:
# --- Configuration ---
DATA_PATH = '/Users/dm954/Documents/code/mixed_diffusion/data/CITEseq/citeseq_preprocessed.h5ad'  # adjust if different
OUTPUT_DIR = 'citeseq_preprocessed_outputs'
GENE_PRESENCE_THRESHOLD = 0.01  # >5% of cells
LABEL_COLUMNS_CANDIDATES = ['cell_labels', 'cell_type1', 'cell_type', 'labels', 'annot', 'CellType', 'celltype.l2']
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: citeseq_preprocessed_outputs


In [23]:
# --- Load AnnData ---
adata = sc.read_h5ad(DATA_PATH)
print(adata)
print('Shape (cells, genes):', adata.shape)
# Inspect available layers/obsm for logging purposes
print('Layers:', list(adata.layers.keys()))
print('Obs columns:', list(adata.obs.columns))

AnnData object with n_obs × n_vars = 10000 × 3000
    obs: 'cell_labels', 'lane'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    obsm: 'protein_expression'
    layers: 'counts'
Shape (cells, genes): (10000, 3000)
Layers: ['counts']
Obs columns: ['cell_labels', 'lane']


In [24]:
# --- Obtain logcounts-like matrix ---
# R script uses precomputed logcounts. In AnnData, often adata.X is already log-normalized (after sc.pp.log1p).
# We'll attempt to find a suitable matrix in layers; fallback to X then perform log1p if needed.
log_layer_candidates = ['logcounts', 'log_norm', 'log']
log_matrix = adata.X

In [25]:
# --- Extract labels (replicating labels1 in R) ---
label_series = adata.obs['cell_labels']
label_df = pd.DataFrame({'x': label_series.values})

In [26]:
# --- Binarize expression (presence/absence) ---
# R: logcounts.dataframe[logcounts.dataframe > 0] = 1
binary_matrix = (log_matrix > 0).astype(np.uint8)
n_cells, n_genes = binary_matrix.shape
print(f'Binary matrix shape: {binary_matrix.shape}')
gene_presence_counts = binary_matrix.sum(axis=0)
print('Gene presence stats: min', gene_presence_counts.min(), 'max', gene_presence_counts.max())
threshold = int(np.ceil(GENE_PRESENCE_THRESHOLD * n_cells))
print(f'Presence threshold (> {GENE_PRESENCE_THRESHOLD*100:.1f}% cells):', threshold)

Binary matrix shape: (10000, 3000)
Gene presence stats: min 3 max 6031
Presence threshold (> 1.0% cells): 100


In [27]:
# --- Filter genes present in >5% cells ---
# R used: rowdata.sum > length(coldata.sum)*0.05 (rowdata.sum was gene sums)
keep_mask = gene_presence_counts > threshold
filtered_matrix = log_matrix[:, keep_mask]  # keep original (non-binary) values like R stored pre-binarized copy
filtered_binary = binary_matrix[:, keep_mask]
filtered_genes = np.array(adata.var_names)[keep_mask]
print('Filtered genes:', len(filtered_genes), '/', n_genes)

Filtered genes: 2269 / 3000


In [28]:
# --- Prepare DataFrames for export ---
cells_index = adata.obs_names
df_filtered = pd.DataFrame(filtered_matrix, index=cells_index, columns=filtered_genes)
df_filtered_binary = pd.DataFrame(filtered_binary, index=cells_index, columns=filtered_genes)
print(df_filtered.shape, df_filtered_binary.shape)
df_filtered.head()

(10000, 2269) (10000, 2269)


,HES4,ISG15,TNFRSF18,TNFRSF4,TAS1R3,MIB2,MMP23B,SMIM1,TNFRSF25,Z98884.1,...,BCAN,VLDLR-AS1,LINC01219,ASPHD1,PRR5-ARHGAP8,AC092807.3,GPD1,ERICH2,AL359551.1,LINC01228
E2L5_TAGACTGTCCAGCAAT,-0.392264,0.757535,-0.221234,-0.273001,-0.037518,0.426373,-0.317100,-0.113256,-0.341156,0.016802,...,-0.017172,-0.010168,0.013166,-0.017321,-0.017369,-0.016690,-0.018839,-0.019802,-0.015823,-0.016228
E2L2_ACAACCATCGCCTTGT,3.144080,-0.177392,-0.197017,-0.165544,-0.040961,-0.361385,-0.032626,-0.108874,-0.336634,-0.033493,...,-0.017138,0.029602,-0.158119,-0.028426,-0.015324,-0.015933,-0.018671,-0.019792,-0.015097,-0.016053
E2L4_CCTACGTAGCAAGGAA,-0.172309,-0.768797,-0.197985,-0.169038,-0.038316,-0.178377,-0.657358,-0.108820,-0.335055,-0.016383,...,-0.017147,0.028562,-0.002520,-0.017733,-0.016016,-0.016752,-0.018970,-0.019820,-0.015961,-0.016346
L2_CCGGACAAGTTGCGCC,-0.220671,-0.576744,-0.223505,-0.223514,-0.037949,-0.276531,-0.080441,-0.125488,-0.407008,-0.008217,...,-0.017152,-0.005246,0.006679,-0.017615,-0.016011,-0.016919,-0.018835,-0.019802,-0.015919,-0.016336
E2L8_TCGCAGGTCGACACCG,-0.165401,-0.287821,-0.754851,-0.141197,-0.038140,0.094149,0.374721,-0.110691,-0.345229,-0.050743,...,-0.017146,0.053000,-0.018082,-0.017639,-0.016032,-0.016765,-0.018692,-0.019795,-0.016044,-0.016364


In [29]:
# --- Save outputs (space-delimited to mimic R script) ---
filtered_csv_path = Path(OUTPUT_DIR) / 'citeseq_preprocessed_log_genefilt.csv'
filtered_txt_path = Path(OUTPUT_DIR) / 'citeseq_preprocessed_log_genefilt.txt'
labels_csv_path = Path(OUTPUT_DIR) / 'citeseq_truelabels1.csv'
labels_txt_path = Path(OUTPUT_DIR) / 'citeseq_truelabels.txt'
df_filtered.to_csv(filtered_csv_path, sep=' ', index=True, header=True)
df_filtered.to_csv(filtered_txt_path, sep=' ', index=False, header=False)
label_df.to_csv(labels_csv_path, sep=' ', index=False, header=False)
label_df.to_csv(labels_txt_path, sep=' ', index=False, header=False)
print('Saved:')
print('  ', filtered_csv_path)
print('  ', filtered_txt_path)
print('  ', labels_csv_path)
print('  ', labels_txt_path)

Saved:
   citeseq_preprocessed_outputs/citeseq_preprocessed_log_genefilt.csv
   citeseq_preprocessed_outputs/citeseq_preprocessed_log_genefilt.txt
   citeseq_preprocessed_outputs/citeseq_truelabels1.csv
   citeseq_preprocessed_outputs/citeseq_truelabels.txt
